# FIFA World Cup Historical Football Matches - Advanced Exploratory Data Analysis (EDA)

This notebook performs an advanced, high-complexity Exploratory Data Analysis (EDA) on a dataset containing approximately 50,000 international football matches from 1872 to 2026. The structure, methods, and feature engineering follow the patterns from the sample folder notebook, using custom iterative Elo ratings as the foundation for team strength. All key analyses are conducted on the **top 20 international teams** to ensure depth and granularity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import poisson
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import PoissonRegressor
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

# Set plotting aesthetics
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["font.size"] = 11

# 1. Load and Clean Dataset
df = pd.read_csv("Dataset/results.csv")
df = df.dropna(subset=['home_score', 'away_score']).reset_index(drop=True)
df['home_score'] = df['home_score'].astype(int)
df['away_score'] = df['away_score'].astype(int)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"Dataset dimensions: {df.shape[0]} matches, {df.shape[1]} columns.")
df.head()

## 1. Custom Elo Rating System Calculation (with Weighted Recent Form)

We calculate Elo ratings iteratively for all teams. We also maintain weighted historical results lists to estimate linear-decay recent forms (where more recent match outcomes are given higher weight).

In [ ]:
all_teams = set(df['home_team']).union(set(df['away_team']))
elo_ratings = {team: 1500.0 for team in all_teams}
recent_results = {team: [] for team in all_teams} 
recent_goals_scored = {team: [] for team in all_teams}
recent_goals_conceded = {team: [] for team in all_teams}
elo_history = []

def get_k_factor(tournament):
    t_lower = tournament.lower()
    if 'friendly' in t_lower:
        return 10
    elif 'fifa world cup' in t_lower and 'qualification' not in t_lower:
        return 60
    elif 'fifa world cup qualification' in t_lower:
        return 40
    elif any(term in t_lower for term in ['qualification', 'nations league', 'euro', 'copa', 'african cup', 'asian cup', 'gold cup']):
        return 30
    else:
        return 20

def get_weighted_mean(values):
    if not values:
        return 0.5
    n = len(values)
    weights = np.arange(1, n + 1)
    return np.sum(np.array(values) * weights) / np.sum(weights)

def get_weighted_mean_goals(values, default):
    if not values:
        return default
    n = len(values)
    weights = np.arange(1, n + 1)
    return np.sum(np.array(values) * weights) / np.sum(weights)

print("Iteratively calculating Elo ratings...")
for idx, row in df.iterrows():
    h_team, a_team = row['home_team'], row['away_team']
    h_score, a_score = row['home_score'], row['away_score']
    tourney, neutral, m_date = row['tournament'], row['neutral'], row['date']
    
    elo_h, elo_a = elo_ratings[h_team], elo_ratings[a_team]
    
    # Record annual samples for history
    elo_history.append({'date': m_date, 'team': h_team, 'elo': elo_h})
    elo_history.append({'date': m_date, 'team': a_team, 'elo': elo_a})
        
    # Expected outcome
    H_val = 100.0 if not neutral else 0.0
    w_e_h = 1.0 / (10.0 ** (-((elo_h + H_val) - elo_a) / 400.0) + 1.0)
    w_e_a = 1.0 - w_e_h
    
    # Actual outcome
    if h_score > a_score:
        w_h, w_a = 1.0, 0.0
    elif h_score == a_score:
        w_h, w_a = 0.5, 0.5
    else:
        w_h, w_a = 0.0, 1.0
        
    gd = abs(h_score - a_score)
    G = 1.0 if gd <= 1 else (1.5 if gd == 2 else (11.0 + gd) / 8.0)
    K = get_k_factor(tourney)
    
    # Update ratings
    elo_ratings[h_team] = elo_h + K * G * (w_h - w_e_h)
    elo_ratings[a_team] = elo_a + K * G * (w_a - w_e_a)
    
    # Update forms
    recent_results[h_team] = (recent_results[h_team] + [w_h])[-5:]
    recent_results[a_team] = (recent_results[a_team] + [w_a])[-5:]
    recent_goals_scored[h_team] = (recent_goals_scored[h_team] + [h_score])[-5:]
    recent_goals_conceded[h_team] = (recent_goals_conceded[h_team] + [a_score])[-5:]
    recent_goals_scored[a_team] = (recent_goals_scored[a_team] + [a_score])[-5:]
    recent_goals_conceded[a_team] = (recent_goals_conceded[a_team] + [h_score])[-5:]

elo_hist_df = pd.DataFrame(elo_history)
print("Elo rating calculations complete.")

## 2. Identify the Top 20 International Teams

We identify the top 20 teams based on their final calculated Elo ratings. All subsequent team-level analyses will focus on these nations.

In [ ]:
top_20_teams = [t[0] for t in sorted(elo_ratings.items(), key=lambda x: x[1], reverse=True)[:20]]
print("Top 20 Teams by Final Elo Rating:")
for rank, team in enumerate(top_20_teams):
    print(f"{rank+1}. {team}: {elo_ratings[team]:.1f}")

## 3. Feature Engineering with Weighted Form & H2H

We extract pre-match features for all games played since 2000, compute decaying forms, and build a symmetrized training set.

In [ ]:
match_features = []
recent_results_f = {team: [] for team in all_teams} 
recent_goals_scored_f = {team: [] for team in all_teams}
recent_goals_conceded_f = {team: [] for team in all_teams}
h2h_records = {}
elo_ratings_f = {team: 1500.0 for team in all_teams}

print("Extracting features for post-2000 matches...")
for idx, row in df.iterrows():
    h_team, a_team = row['home_team'], row['away_team']
    h_score, a_score = row['home_score'], row['away_score']
    neutral, m_date, tourney = row['neutral'], row['date'], row['tournament']
    
    elo_h, elo_a = elo_ratings_f[h_team], elo_ratings_f[a_team]
    
    form_h = get_weighted_mean(recent_results_f[h_team])
    form_a = get_weighted_mean(recent_results_f[a_team])
    scored_h = get_weighted_mean_goals(recent_goals_scored_f[h_team], 1.5)
    conceded_a = get_weighted_mean_goals(recent_goals_conceded_f[a_team], 1.5)
    
    pair = tuple(sorted([h_team, a_team]))
    if pair not in h2h_records:
        h2h_records[pair] = {'matches': 0, 'wins': {h_team: 0, a_team: 0}}
    n_h2h = h2h_records[pair]['matches']
    h2h_win_h = h2h_records[pair]['wins'].get(h_team, 0) / n_h2h if n_h2h > 0 else 0.5
    h2h_win_a = h2h_records[pair]['wins'].get(a_team, 0) / n_h2h if n_h2h > 0 else 0.5
    
    if m_date >= pd.to_datetime('2000-01-01'):
        match_features.append({
            'elo_team': elo_h,
            'elo_opp': elo_a,
            'elo_diff': elo_h - elo_a,
            'form_team': form_h,
            'form_opp': form_a,
            'form_team_scored': scored_h,
            'form_opp_conceded': conceded_a,
            'h2h_win_team': h2h_win_h,
            'h2h_win_opp': h2h_win_a,
            'is_home': 0 if neutral == 1 else 1,
            'goals_team': h_score,
            'goals_opp': a_score,
            'outcome': 2 if h_score > a_score else (1 if h_score == a_score else 0)
        })
        match_features.append({
            'elo_team': elo_a,
            'elo_opp': elo_h,
            'elo_diff': elo_a - elo_h,
            'form_team': form_a,
            'form_opp': form_h,
            'form_team_scored': get_weighted_mean_goals(recent_goals_scored_f[a_team], 1.2),
            'form_opp_conceded': get_weighted_mean_goals(recent_goals_conceded_f[h_team], 1.2),
            'h2h_win_team': h2h_win_a,
            'h2h_win_opp': h2h_win_h,
            'is_home': 0 if neutral == 1 else -1,
            'goals_team': a_score,
            'goals_opp': h_score,
            'outcome': 0 if h_score > a_score else (1 if h_score == a_score else 2)
        })
        
    w_h = 1.0 if h_score > a_score else (0.5 if h_score == a_score else 0.0)
    w_a = 1.0 - w_h
    recent_results_f[h_team] = (recent_results_f[h_team] + [w_h])[-5:]
    recent_results_f[a_team] = (recent_results_f[a_team] + [w_a])[-5:]
    recent_goals_scored_f[h_team] = (recent_goals_scored_f[h_team] + [h_score])[-5:]
    recent_goals_conceded_f[h_team] = (recent_goals_conceded_f[h_team] + [a_score])[-5:]
    recent_goals_scored_f[a_team] = (recent_goals_scored_f[a_team] + [a_score])[-5:]
    recent_goals_conceded_f[a_team] = (recent_goals_conceded_f[a_team] + [h_score])[-5:]
    
    H_val = 100.0 if not neutral else 0.0
    w_e_h = 1.0 / (10.0 ** (-((elo_h + H_val) - elo_a) / 400.0) + 1.0)
    w_e_a = 1.0 - w_e_h
    gd = abs(h_score - a_score)
    G = 1.0 if gd <= 1 else (1.5 if gd == 2 else (11.0 + gd) / 8.0)
    K = get_k_factor(tourney)
    elo_ratings_f[h_team] = elo_h + K * G * (w_h - w_e_h)
    elo_ratings_f[a_team] = elo_a + K * G * (w_a - w_e_a)
    
    h2h_records[pair]['matches'] += 1
    if w_h == 1.0:
        h2h_records[pair]['wins'][h_team] += 1
    elif w_a == 1.0:
        h2h_records[pair]['wins'][a_team] += 1

features_df = pd.DataFrame(match_features)
print(f"Created {features_df.shape[0]} symmetrized post-2000 match records.")

## 4. Feature Correlation Matrix Heatmap

We calculate and display a heatmap of features to explore correlations and multicollinearity.

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(features_df.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix of Engineered Features')
plt.tight_layout()
plt.show()

## 5. Model Optimization and GridSearchCV Hyperparameter Tuning

Following the sample notebook's complexity, we perform hyperparameter grid-searches to fit the Random Forest outcome classifier.

In [ ]:
X_clf = features_df[['elo_team', 'elo_opp', 'elo_diff', 'form_team', 'form_opp', 'h2h_win_team', 'h2h_win_opp', 'is_home']]
y_clf = features_df['outcome']

param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [8, 12],
    'min_samples_split': [2, 5]
}

print("Tuning RandomForestClassifier...")
grid_search = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), param_grid, cv=3, scoring='accuracy')
grid_search.fit(X_clf, y_clf)

best_clf = grid_search.best_estimator_
print("Best parameters:", grid_search.best_params_)
print(f"Cross-validated training accuracy: {grid_search.best_score_ * 100:.2f}%")

## 6. Model Feature Importances

We visualize the relative importance of each engineered feature in predicting match outcomes.

In [ ]:
importances = best_clf.feature_importances_
features = X_clf.columns
indices = np.argsort(importances)

plt.figure(figsize=(10, 6))
plt.barh(range(len(indices)), importances[indices], color='#3b82f6', align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Relative Feature Importance')
plt.title('Random Forest Feature Importance Analysis')
plt.tight_layout()
plt.show()

## 7. Poisson Regressor Tuning and Fitting

We scale Elo rating variables (dividing by 400) to prevent overflow in the GLM solver, and use cross-validation to choose the L2 regularization parameter.

In [ ]:
features_df['elo_team_scaled'] = features_df['elo_team'] / 400.0
features_df['elo_opp_scaled'] = features_df['elo_opp'] / 400.0
features_df['elo_diff_scaled'] = features_df['elo_diff'] / 400.0

X_poi = features_df[['elo_team_scaled', 'elo_opp_scaled', 'elo_diff_scaled', 'form_team_scored', 'form_opp_conceded', 'is_home']]
y_poi = features_df['goals_team']

param_grid_poi = {'alpha': [1e-6, 1e-4, 1e-2]}
grid_poi = GridSearchCV(PoissonRegressor(max_iter=300), param_grid_poi, cv=3)
grid_poi.fit(X_poi, y_poi)

best_poi = grid_poi.best_estimator_
print("Poisson Regressor best alpha:", grid_poi.best_params_)
print("Poisson Regressor coefficients:", best_poi.coef_)

## 8. Historical Elo Rating Trajectories (Top 20 Teams)

We visualize the Elo trajectories of the top 20 teams dynamically over time.

In [ ]:
elo_hist_df['year'] = elo_hist_df['date'].dt.year
elo_annual = elo_hist_df.groupby(['year', 'team'])['elo'].last().reset_index()
elo_top_20 = elo_annual[elo_annual['team'].isin(top_20_teams)]

plt.figure(figsize=(16, 10))
sns.lineplot(data=elo_top_20, x='year', y='elo', hue='team', linewidth=1.5)
plt.title('Elo Rating Trajectories of Top 20 International Teams (1872 - 2026)')
plt.xlabel('Year')
plt.ylabel('Elo Rating')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Top 20 Teams')
plt.tight_layout()
plt.show()

## 9. Historical Pairwise Head-to-Head Win Rate Matrix (Top 20 Teams)

We display the pairwise win ratios of matches played between the top 20 teams.

In [ ]:
h2h_matrix = pd.DataFrame(np.nan, index=top_20_teams, columns=top_20_teams)

for t1 in top_20_teams:
    for t2 in top_20_teams:
        if t1 == t2:
            h2h_matrix.loc[t1, t2] = 0.5
            continue
        pair = tuple(sorted([t1, t2]))
        if pair in h2h_records and h2h_records[pair]['matches'] > 0:
            h2h_matrix.loc[t1, t2] = h2h_records[pair]['wins'].get(t1, 0) / h2h_records[pair]['matches']
            
plt.figure(figsize=(16, 12))
sns.heatmap(h2h_matrix, annot=True, cmap="YlOrRd", fmt=".2f")
plt.title('Historical Pairwise Head-to-Head Win Ratios (Top 20 Teams)')
plt.tight_layout()
plt.show()

## 10. Goals Trends & Home Advantage Analysis

We compare historical goal-scoring trends and home venue advantage distributions.

In [ ]:
df['year'] = df['date'].dt.year
df['total_goals'] = df['home_score'] + df['away_score']
goal_trend = df.groupby('year')['total_goals'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.lineplot(data=goal_trend, x='year', y='total_goals', ax=axes[0], color='#fbbf24', linewidth=2)
axes[0].set_title('Average Goals Scored per Match Over Time')

df['outcome'] = df.apply(lambda r: 'Home Win' if r['home_score'] > r['away_score'] else ('Away Win' if r['home_score'] < r['away_score'] else 'Draw'), axis=1)
venue_stats = df.groupby('neutral')['outcome'].value_counts(normalize=True).unstack() * 100
venue_stats.plot(kind='bar', stacked=True, ax=axes[1], color=['#ef4444', '#94a3b8', '#10b981'])
axes[1].set_title('Outcome Shares: Home vs Neutral Venues')
axes[1].set_xticklabels(['Home Venue', 'Neutral Venue'], rotation=0)
plt.tight_layout()
plt.show()

## 11. Goals Distribution Validation

We check that goals scored in international games approximate a theoretical Poisson distribution.

In [ ]:
all_team_goals = pd.concat([df['home_score'], df['away_score']])
mean_goals = all_team_goals.mean()
max_goals = 8
actual_freqs = all_team_goals.value_counts(normalize=True).reindex(range(max_goals+1), fill_value=0)
poisson_probs = [poisson.pmf(k, mean_goals) for k in range(max_goals+1)]

x = np.arange(max_goals+1)
width = 0.35
plt.bar(x - width/2, actual_freqs, width, label='Actual Match Goals', color='#3b82f6')
plt.bar(x + width/2, poisson_probs, width, label='Poisson Theoretical', color='#fbbf24', alpha=0.8)
plt.title('Validation: Goals Scored vs Theoretical Poisson Distribution')
plt.xticks(x)
plt.legend()
plt.tight_layout()
plt.show()

## 12. 2026 FIFA World Cup - Single Bracket Walkthrough Simulation

We simulate a complete 2026 FIFA World Cup tournament run (48 teams), showing group stage results and knockout round advancement steps with scores, mirroring the tournament simulation printed logs of the sample notebook.

In [ ]:
groups = {
    'A': ['Mexico', 'South Africa', 'South Korea', 'Czech Republic'],
    'B': ['Canada', 'Bosnia & Herzegovina', 'Qatar', 'Switzerland'],
    'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'D': ['United States', 'Paraguay', 'Australia', 'Türkiye'],
    'E': ['Germany', 'Curaçao', 'Ivory Coast', 'Ecuador'],
    'F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'H': ['Spain', 'Cabo Verde', 'Saudi Arabia', 'Uruguay'],
    'I': ['France', 'Senegal', 'Iraq', 'Norway'],
    'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'],
    'L': ['England', 'Croatia', 'Ghana', 'Panama']
}

poi_coef = best_poi.coef_
poi_intercept = best_poi.intercept_

def get_elo_from_dict(t):
    NAME_MAP_TO_DATASET = {
        'Bosnia & Herzegovina': 'Bosnia and Herzegovina',
        'Cabo Verde': 'Cape Verde',
        'Türkiye': 'Turkey'
    }
    t_ds = NAME_MAP_TO_DATASET.get(t, t)
    return elo_ratings.get(t_ds, 1500.0)

def get_sim_stats(t):
    NAME_MAP_TO_DATASET = {'Bosnia & Herzegovina': 'Bosnia and Herzegovina', 'Cabo Verde': 'Cape Verde', 'Türkiye': 'Turkey'}
    t_ds = NAME_MAP_TO_DATASET.get(t, t)
    w_pts = get_weighted_mean(recent_results.get(t_ds, []))
    s_goals = get_weighted_mean_goals(recent_goals_scored.get(t_ds, []), 1.5)
    c_goals = get_weighted_mean_goals(recent_goals_conceded.get(t_ds, []), 1.5)
    return get_elo_from_dict(t), w_pts, s_goals, c_goals

def sim_goals(t1, t2):
    elo_t1, _, scored_t1, conceded_t1 = get_sim_stats(t1)
    elo_t2, _, scored_t2, conceded_t2 = get_sim_stats(t2)
    log_lambda_1 = (elo_t1/400)*poi_coef[0] + (elo_t2/400)*poi_coef[1] + ((elo_t1-elo_t2)/400)*poi_coef[2] + scored_t1*poi_coef[3] + conceded_t2*poi_coef[4] + poi_intercept
    log_lambda_2 = (elo_t2/400)*poi_coef[0] + (elo_t1/400)*poi_coef[1] + ((elo_t2-elo_t1)/400)*poi_coef[2] + scored_t2*poi_coef[3] + conceded_t1*poi_coef[4] + poi_intercept
    return np.random.poisson(max(0.01, np.exp(log_lambda_1))), np.random.poisson(max(0.01, np.exp(log_lambda_2)))

def sim_knockout(t1, t2):
    g1, g2 = sim_goals(t1, t2)
    if g1 > g2:
        return t1, g1, g2
    elif g2 > g1:
        return t2, g1, g2
    else:
        winner = t1 if get_elo_from_dict(t1) > get_elo_from_dict(t2) else t2
        return winner, g1, g2

print("=== SIMULATING 2026 WORLD CUP GROUP STAGE ===\n")
group_results = {}
for letter, teams in groups.items():
    points = {t: 0 for t in teams}
    gd = {t: 0 for t in teams}
    goals = {t: 0 for t in teams}
    
    # Play all matches in group
    for i in range(len(teams)):
        for j in range(i+1, len(teams)):
            t1, t2 = teams[i], teams[j]
            g1, g2 = sim_goals(t1, t2)
            goals[t1] += g1
            goals[t2] += g2
            gd[t1] += (g1 - g2)
            gd[t2] += (g2 - g1)
            if g1 > g2:
                points[t1] += 3
            elif g2 > g1:
                points[t2] += 3
            else:
                points[t1] += 1
                points[t2] += 1
                
    ranked = sorted(teams, key=lambda t: (points[t], gd[t], goals[t]), reverse=True)
    group_results[letter] = ranked
    print(f"Group {letter} Standings:")
    for rank, team in enumerate(ranked):
        print(f"  {rank+1}. {team} - Pts: {points[team]}, GD: {gd[team]}, GF: {goals[team]}")
    print()

# Knockout setup (Round of 32)
print("=== SIMULATING KNOCKOUT ROUND OF 32 ===\n")
r32_matches = [
    (group_results['A'][0], group_results['B'][1]),
    (group_results['B'][0], group_results['A'][1]),
    (group_results['C'][0], group_results['D'][2]),
    (group_results['D'][0], group_results['E'][1]),
    (group_results['E'][0], group_results['D'][1]),
    (group_results['F'][0], group_results['E'][2]),
    (group_results['G'][0], group_results['H'][1]),
    (group_results['H'][0], group_results['G'][1]),
    (group_results['I'][0], group_results['H'][2]),
    (group_results['J'][0], group_results['K'][1]),
    (group_results['K'][0], group_results['J'][1]),
    (group_results['L'][0], group_results['K'][2]),
    (group_results['C'][1], group_results['A'][2]),
    (group_results['F'][1], group_results['B'][2]),
    (group_results['I'][1], group_results['G'][2]),
    (group_results['L'][1], group_results['I'][2])
]

r16_teams = []
for t1, t2 in r32_matches:
    winner, g1, g2 = sim_knockout(t1, t2)
    r16_teams.append(winner)
    print(f"  {t1} {g1} - {g2} {t2} => Winner: {winner}")
print()

print("=== SIMULATING ROUND OF 16 ===\n")
qf_teams = []
for i in range(0, len(r16_teams), 2):
    t1, t2 = r16_teams[i], r16_teams[i+1]
    winner, g1, g2 = sim_knockout(t1, t2)
    qf_teams.append(winner)
    print(f"  {t1} {g1} - {g2} {t2} => Winner: {winner}")
print()

print("=== SIMULATING QUARTER-FINALS ===\n")
sf_teams = []
for i in range(0, len(qf_teams), 2):
    t1, t2 = qf_teams[i], qf_teams[i+1]
    winner, g1, g2 = sim_knockout(t1, t2)
    sf_teams.append(winner)
    print(f"  {t1} {g1} - {g2} {t2} => Winner: {winner}")
print()

print("=== SIMULATING SEMI-FINALS ===\n")
final_teams = []
for i in range(0, len(sf_teams), 2):
    t1, t2 = sf_teams[i], sf_teams[i+1]
    winner, g1, g2 = sim_knockout(t1, t2)
    final_teams.append(winner)
    print(f"  {t1} {g1} - {g2} {t2} => Winner: {winner}")
print()

print("=== SIMULATING THE FIFA WORLD CUP 2026 FINAL ===\n")
t1, t2 = final_teams[0], final_teams[1]
champion, g1, g2 = sim_knockout(t1, t2)
print(f"  🏆 {t1} {g1} - {g2} {t2} => CHAMPION: {champion} 🏆")
